# Extended-Reach Horizontal Well Kill Schedule Analysis
## Comparing True Vertical Depth (TVD) Mud-Front Tracking vs. Naive Linear Stroke Step-Down

### Executive Summary
In conventional vertical wells, drilling mud displacement strokes map linearly to True Vertical Depth (TVD), making a simple linear step-down from ICP to FCP valid. 

In **deviated and extended-reach horizontal wells**, this assumption completely breaks down:
1. In the build section, the vertical descent rate per stroke continually slows as inclination increases.
2. Across the lateral (from heel to toe), **TVD is constant** ($0\text{ ft/ft}$ vertical gain). Hydrostatic gain halts entirely.
3. Concurrently, as heavier kill mud replaces original mud along thousands of feet of horizontal pipe, **circulating friction steadily increases**.

This case study demonstrates why a linear schedule causes severe downhole overbalance or underbalance, and demonstrates the exact physics of the depth-based step-down schedule.

In [ ]:
from killsheet.schedule import pressure_schedule, TEN_STEPS
from killsheet.formulas import kill_mud_weight, initial_circulating_pressure, final_circulating_pressure

# Horizontal Well Geometry (Wolfcamp Lateral)
# KOP @ 6,500 ft MD/TVD; End of Build / Heel @ 9,500 ft MD / 9,000 ft TVD; Bit @ 19,500 ft MD / 9,000 ft TVD
key_points = [
    ("KOP", 6500, 6500),
    ("Heel", 9500, 9000),
    ("Bit", 19500, 9000),
]

drill_string = [
    (0.0178, 18500),   # 18,500 ft of 5" DP (0.0178 bbl/ft)
    (0.0087, 700),     # 700 ft of HWDP
    (0.0077, 300),     # 300 ft of Drill Collars
]

omw_ppg = 10.5
sidpp_psi = 550
scr_pressure_psi = 600
pump_output_bbl_per_stk = 0.117

### 1. Kill Parameters
Calculate KMW, ICP, and FCP based on bit TVD (9,000 ft), NOT bit MD (19,500 ft).

In [ ]:
kmw = kill_mud_weight(sidpp_psi, tvd_ft=9000, original_mud_weight_ppg=omw_ppg)
icp = initial_circulating_pressure(sidpp_psi, scr_pressure_psi)
fcp = final_circulating_pressure(scr_pressure_psi, kmw, omw_ppg)

print(f"Kill Mud Weight (KMW): {kmw:.1f} ppg")
print(f"Initial Circulating Pressure (ICP): {icp} psi")
print(f"Final Circulating Pressure (FCP):   {fcp} psi")

### 2. Computing the True Step-Down Schedule
We generate the 10-step schedule accounting for the exact location of the kill mud front.

In [ ]:
schedule = pressure_schedule(
    icp_psi=icp,
    fcp_psi=fcp,
    drill_string_sections=drill_string,
    pump_output_bbl_per_stk=pump_output_bbl_per_stk,
    sidpp_psi=sidpp_psi,
    bit_tvd_ft=9000,
    key_points=key_points,
    step_method=TEN_STEPS,
)

print(f"{'Strokes':>8} {'MD (ft)':>9} {'TVD (ft)':>9} {'DP Pressure':>13} {'Label':<15}")
print("-" * 58)
for row in schedule:
    print(f"{row.strokes:>8,} {row.md_ft:>9,} {row.tvd_ft:>9,} {row.pressure_psi:>10,} psi   {row.label:<15}")

### 3. Engineering Analysis: The Lateral Pressure Rebound
Notice the drill pipe pressure trajectory:
1. **Surface to KOP (0 to 6,500 ft):** DP pressure steps down steadily as kill mud gains vertical depth.
2. **KOP to Heel (6,500 to 9,500 ft):** Pressure reaches its absolute minimum at the **Heel**.
3. **Heel to Bit (9,500 to 19,500 ft):** As kill mud enters the horizontal lateral, TVD remains constant at 9,000 ft. **No further hydrostatic head is gained**, but heavier mud fills the lateral, increasing pipe friction. 

To maintain constant bottomhole pressure, the choke operator must allow drill pipe pressure to **climb back up from the heel to FCP** at the bit. A naive straight line would starve the well of backpressure, swabbing the lateral underbalanced.